In [22]:
from torch_geometric.data import HeteroData
import torch
import pickle
import numpy as np
import torch

In [23]:
PTH = r'.\picklefiles'

with open(f'{PTH}\\eal_gnn_trainsmall.pkl', 'rb') as f:
    data = pickle.load(f)


In [75]:
data[1][0]

{'o_id': 'bc55ad6b125f568076ed15b53235993b',
 'id': '1',
 'target_entity': 'Gautama Buddha',
 'sentence': 'The iconography of Gautama Buddha in Laos and Thailand recall specific episodes during his travels and teachings that are familiar to the Buddhists according to an iconography with specific rules.',
 'entities': [{'eid': '10',
   'original_id': 'enwiki:Gautama%20Buddha',
   'entity': 'Gautama Buddha',
   'mention': 'Gautama Buddha'},
  {'eid': '11',
   'original_id': 'enwiki:Laos',
   'entity': 'Laos',
   'mention': 'Laos'},
  {'eid': '12',
   'original_id': 'enwiki:Thailand',
   'entity': 'Thailand',
   'mention': 'Thailand'},
  {'eid': '13',
   'original_id': 'enwiki:Episode',
   'entity': 'Episode',
   'mention': 'episode'},
  {'eid': '15',
   'original_id': 'enwiki:Iconography',
   'entity': 'Iconography',
   'mention': 'iconography'}]}

In [43]:
def get_node_id(data, subject = 'target_entity'):
    res = []
    dic = {}
    if subject == 'target_entity':
        ent = [data[i][0] for i in range(len(data))]
        return list(np.arange(len(ent)))
    
    elif subject == 't_entities':
        j = 0
        ent = [data[i][0] for i in range(len(data))]
        for el in ent:
            for e in el['entities']:
                if e['original_id'] not in dic:
                    dic[e['original_id']] = j
                    j += 1
        return dic
    
    elif subject == 'aspect_entity':
        asp = [data[i][1] for i in range(len(data))]
        j = 0
        for el in asp:
            tasp_id = el['true_aspect_id']
            if tasp_id not in dic:
                dic[tasp_id] = j
                j += 1
            for cand in el['candidate_aspects']:
                asp_id = cand['aspect_id']
                if asp_id != tasp_id:
                    if asp_id not in dic:
                        dic[asp_id] = j
                        j += 1
        return dic
    
    elif subject == 'a_entities':
        asp = [data[i][1] for i in range(len(data))]
        j = 0
        for item in asp:
            for el in item['candidate_aspects']:
                for ent in el['entities']:
                    if ent['original_id'] not in dic:
                        dic[ent['original_id']] = j
                        j += 1
        return dic

In [85]:
def create_targetedge_index(data, asp_id):
    edge = []
    for i in range(len(data)):
        _, asp = data[i]
        tasp_id = asp['true_aspect_id']
        edge.append((i, asp_id[tasp_id]))
    return torch.tensor(edge).T

In [89]:
def create_associated_edge_index(data, subject = 'target_entity', **kwargs):
    edge = []
    if subject == 'target_entity':
        for i in range(len(data)):
            ent, _  = data[i]
            for item in ent['entities']:
                edge.append((i, associated_entity_dict[item['original_id']]))
        edge = list(set(edge))
        return torch.tensor(edge).T
    
    elif subject == 'aspect_entity':
        for i in range(len(data)):
            _, asp = data[i]
            for element in asp['candidate_aspects']:
                first = asp_dict[element['aspect_id']]
                for ent in element['entities']:
                    edge.append((first, ent_asp_id_dict[ent['original_id']]))
        return torch.tensor(edge).T

In [90]:
def read_pickle(name):
    with open(f'{PTH}\\{name}', 'rb') as f:
        data = pickle.load(f)
    f.close()
    return data

In [91]:
target_entity_id = torch.tensor(get_node_id(data))
associated_entity_id = torch.tensor(list(get_node_id(data, 't_entities').values()))

In [92]:
asp_id = torch.tensor(list(get_node_id(data, 'aspect_entity').values()))
ent_asp_id = torch.tensor(list(get_node_id(data, 'a_entities').values()))

In [94]:
asp_dict = get_node_id(data, 'aspect_entity')
target_edge_index = create_targetedge_index(data, asp_dict)

In [95]:
associated_entity_dict = get_node_id(data, 't_entities')
ent_asp_id_dict = get_node_id(data, 'a_entities')

associated_tentity_edge_index = create_associated_edge_index(data, associated_entity_dict = associated_entity_dict)
associated_aspentity_edge_index = create_associated_edge_index(data, subject = 'aspect_entity', asp_dict = asp_dict, ent_asp_id_dict = ent_asp_id_dict)

In [96]:
targetent_emb = read_pickle('targetentemb_sent_gnn_trainsmall.pkl')
aspemb = read_pickle('aspemb_sent_gnn_trainsmall.pkl')
ass_ent_emb = read_pickle('associated_entemb_gnn_trainsmall.pkl')
ass_asp_emb = read_pickle('associated_aspentemb_gnn_trainsmall.pkl')
context_ent = read_pickle('sentemb_gnn_trainsmall.pkl')

In [97]:
aspdict = read_pickle('aspembdict_sent_gnn_trainsmall.pkl')

In [87]:
graph = HeteroData()

graph['target_entity'].num_nodes = len(target_entity_id)
graph['target_entity'].node_id = target_entity_id
graph['target_entity'].x = targetent_emb
graph['target_entity'].y = context_ent

graph['t_entities'].num_nodes = len(associated_entity_id)
graph['t_entities'].node_id = associated_entity_id
graph['t_entities'].x = ass_ent_emb

graph['aspect'].num_nodes = len(asp_id)
graph['aspect'].node_id = asp_id
graph['aspect'].x = aspemb

graph['a_entities'].num_nodes = len(ent_asp_id)
graph['a_entities'].node_id = ent_asp_id
graph['a_entities'].x = ass_asp_emb


graph['target_entity', 'linked_to', 'aspect'] = target_edge_index
graph['target_entity', 'associated_to', 't_entities'] = associated_tentity_edge_index
graph['aspect', 'associated_to', 'a_entities'] = associated_aspentity_edge_index

In [100]:
graph

HeteroData(
  (target_entity, linked_to, aspect)=[2, 5498],
  (target_entity, associated_to, t_entities)=[2, 13535],
  (aspect, associated_to, a_entities)=[2, 542835],
  target_entity={
    num_nodes=5498,
    node_id=[5498],
    x=[5498, 150],
    y=[5498, 384]
  },
  t_entities={
    num_nodes=7698,
    node_id=[7698],
    x=[7698, 150]
  },
  aspect={
    num_nodes=5688,
    node_id=[5688],
    x=[5686, 150]
  },
  a_entities={
    num_nodes=53858,
    node_id=[53858],
    x=[53858, 150]
  }
)

In [99]:
with open(f'{PTH}\\graph_trainsmall_linkpred.pkl', 'wb') as f:
    pickle.dump(graph, f)
    print('Dumped')
f.close()

Dumped
